# 01-Rapid Prototyping (Gradio)

Welcome to **Section 12: REST APIs, FastAPI & AI Prototyping**. Over the past 11 sections, you have mastered the raw mathematics of Deep Learning, Computer Vision, and Natural Language Processing. You can build 70-billion parameter neural networks in PyTorch.

But there is a harsh reality in Enterprise AI: **Stakeholders cannot run Jupyter Notebooks.** If you want funding, approval, or user feedback, you must wrap your complex tensor mathematics inside a graphical User Interface (UI).

In this lesson, we will master **Gradio**, an open-source Python library that allows us to rapidly bridge the gap between GPU memory (PyTorch Tensors) and the Web Browser (HTML/JavaScript) in less than 20 lines of code.

Let's set up our PyTorch and Gradio environment to launch our first machine-to-machine interface.

In [2]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import gradio as gr
import numpy as np
import time
from PIL import Image

print("✅ PyTorch & Gradio Web Prototyping Environment Ready.")

✅ PyTorch & Gradio Web Prototyping Environment Ready.


# 1. The Physics of the Web-to-Tensor Bridge

Before we write UI code, we must understand the data engineering required to move data from a user's web browser into a PyTorch model.

When a user clicks "Upload Image" on a website, the image is physically stored on their hard drive as a compressed JPEG or PNG file.

1. **Serialization**: The browser converts this file into a stream of raw bytes (or a Base64 encoded string) and sends it over the internet via HTTP.
2. **Deserialization (Gradio)**: The Gradio server receives the HTTP request and automatically decodes the bytes into a standard Python format. By default, for images, Gradio outputs a **NumPy Array** of shape $[H, W, C]$ with values ranging from $0$ to $255$.
3. **The Tensor Transformation**: PyTorch models strictly require a 4D Tensor of shape $[B, C, H, W]$ with normalized floating-point values between $0.0$ and $1.0$ (or specific ImageNet means/stds).

Your Gradio function is not just a UI element; it is the critical **Mathematical Translation Layer** between the web and the GPU.

# 2. Architecting the Core `gr.Interface`

The most fundamental class in Gradio is `gr.Interface`. It requires exactly three arguments to compile a fully functional React.js web application:

1. `fn`: The Python inference function.
2. `inputs`: The UI component the user interacts with (e.g., `gr.Image()`, `gr.Textbox()`).
3. `outputs`: The UI component that displays the model's prediction.

Let's build a Computer Vision endpoint. We will write an inference function that takes a NumPy image from the web browser, converts it to a PyTorch tensor, applies a mathematical edge-detection convolution, converts it back to NumPy, and returns it to the UI.

In [4]:
# 1. Define the Mathematical Translation & Inference Layer
def vision_inference_pipeline(input_numpy_array):
    """
    Translates Web UI data (NumPy) -> GPU Data (Tensor) -> Web UI Data (NumPy)
    """
    # A. Safety Check
    if input_numpy_array is None:
        return None
        
    # B. Web-to-Tensor Translation
    # [H, W, C] -> [C, H, W]
    tensor_img = torch.tensor(input_numpy_array, dtype=torch.float32).permute(2, 0, 1)
    tensor_img = tensor_img / 255.0 # Normalize to [0.0, 1.0]
    tensor_img = tensor_img.unsqueeze(0) # Add Batch Dimension: [1, C, H, W]
    
    # C. Execute GPU Mathematics (Simulated Edge Detection via Convolution)
    # We apply a simple generic filter using PyTorch functional tools
    weight = torch.tensor([[[[-1., -1., -1.],
                             [-1.,  8., -1.],
                             [-1., -1., -1.]]]])
    
    # Expand weight for 3 RGB channels: [3, 1, 3, 3]
    weight = weight.repeat(3, 1, 1, 1)
    
    with torch.no_grad():
        # Apply depthwise convolution
        edges = F.conv2d(tensor_img, weight, padding=1, groups=3)
        edges = torch.clamp(edges, 0.0, 1.0) # Prevent mathematical overflow
        
    # D. Tensor-to-Web Translation
    # Remove batch, permute back to [H, W, C], convert to uint8 [0, 255]
    output_tensor = edges.squeeze(0).permute(1, 2, 0)
    output_numpy = (output_tensor.numpy() * 255).astype(np.uint8)
    
    return output_numpy

# 2. Architect the UI
print("--- 🌐 Compiling Gradio Interface ---")
vision_app = gr.Interface(
    fn=vision_inference_pipeline,
    inputs=gr.Image(label="Upload Image (Web Browser)"),
    outputs=gr.Image(label="Edge Detection Tensor (GPU Output)"),
    title="Computer Vision Prototyper",
    description="Upload a standard RGB image. The backend will automatically convert it to a PyTorch [1, 3, H, W] tensor, apply an algorithmic convolution, and stream the resulting matrix back to your browser.",
)

# 3. Launch the Server (Commented out so it doesn't block notebook execution)
# vision_app.launch(share=False) 
print("Success! If `vision_app.launch()` is called, a local web server boots up on http://127.0.0.1:7860")

--- 🌐 Compiling Gradio Interface ---
Success! If `vision_app.launch()` is called, a local web server boots up on http://127.0.0.1:7860


# 3. Token Streaming & Generator Functions (`yield`)

In Section 11, we learned that Large Language Models (LLMs) are **Autoregressive**. They predict one word at a time.

If a model takes 5 seconds to generate a 500-word paragraph, and your UI waits until the entire 500 words are finished before returning the string, the user will stare at a frozen loading spinner and assume the app crashed.

To solve this, modern AI interfaces use **Token Streaming**. Instead of a standard `return` statement, our Python function must be a **Generator** using the `yield` keyword. Every time the GPU predicts a new token, we immediately yield it over the HTTP connection to the browser, creating the classic "typing" effect.

# 4. Architecting `gr.Blocks` and Chat Interfaces

`gr.Interface` is great for simple Input $\rightarrow$ Output apps. But Enterprise prototypes require tabs, sidebars, state management, and chat histories. For this, we use the `gr.Blocks` context manager.

Let's build a highly advanced, streaming NLP Chatbot interface.

In [5]:
# 1. Define the Generative AI Streaming Pipeline
def llm_streaming_simulator(user_message, chat_history):
    """
    Simulates an Autoregressive LLM generating text token-by-token.
    Gradio ChatInterfaces expect the function to yield the *entire updated string* at each step.
    """
    # Define a mock response
    simulated_tokens = [
        "The ", "mathematics ", "of ", "Transformers ", "rely ", "heavily ", 
        "on ", "scaled ", "dot-product ", "attention. ", "By ", "using ", 
        "Query, ", "Key, ", "and ", "Value ", "matrices, ", "we ", "can ", 
        "process ", "text ", "in ", "parallel!"
    ]
    
    response_so_far = ""
    
    # Iterate through the sequence, yielding the updated state
    for token in simulated_tokens:
        time.sleep(0.05) # Simulate GPU inference latency
        response_so_far += token
        
        # 'yield' pushes the current state of the string to the frontend UI instantly
        yield response_so_far

# 2. Architect the Enterprise UI Layout
print("--- 🏗️ Compiling Advanced Blocks Layout ---")
with gr.Blocks(theme=gr.themes.Monochrome()) as enterprise_prototype:
    
    gr.Markdown("# 🧠 LLM Autoregressive Streaming Prototype")
    gr.Markdown("This interface demonstrates asynchronous token yielding via Python generators.")
    
    with gr.Row():
        # Left Column: The Chatbot UI
        with gr.Column(scale=3):
            # gr.ChatInterface automatically handles conversation state (memory)
            chat_engine = gr.ChatInterface(
                fn=llm_streaming_simulator,
                chatbot=gr.Chatbot(height=400, layout="bubble"),
                textbox=gr.Textbox(placeholder="Ask the simulated model a question...", container=False)
            )
            
        # Right Column: Model Hyperparameters
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Inference Parameters")
            temp_slider = gr.Slider(minimum=0.1, maximum=2.0, value=0.7, step=0.1, label="Temperature")
            top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.95, step=0.05, label="Top-p (Nucleus)")
            max_tokens = gr.Number(value=512, label="Max Tokens")
            
            # Note: To actively pass these hyperparameters into the ChatInterface, 
            # we would map them using the `additional_inputs` argument in gr.ChatInterface!

# 3. Launch the Server
# enterprise_prototype.launch()
print("Success! The Blocks architecture successfully compiled a React.js dashboard with interactive sliders and a stateful chat history.")

--- 🏗️ Compiling Advanced Blocks Layout ---


/tmp/ipykernel_25496/878335435.py:27: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as enterprise_prototype:


Success! The Blocks architecture successfully compiled a React.js dashboard with interactive sliders and a stateful chat history.


## Real-World Use Case or Analogy:

Think of the transition from a Jupyter Notebook to a Gradio Web App like **Opening a high-end restaurant**:

* **The Jupyter Notebook (The Test Kitchen)**: You, the Head Chef (Machine Learning Engineer), are in a private test kitchen. You are mixing raw ingredients (Tensors), calculating exact baking temperatures (Hyperparameters), and tweaking the recipe (Backpropagation). The food is incredible, but there are no tables, no menus, and no waiters. If an investor wants to taste the food, they have to stand in the kitchen, put on a hairnet, and eat out of a mixing bowl.
* **Gradio (The Restaurant Dining Room)**: Gradio allows you to instantly construct a beautiful dining room with tables, menus, and waiters in 10 minutes. The user sits at a clean table (The Web Browser). They point at the menu (Input). The waiter (The Gradio HTTP Server) takes the order, walks into the kitchen, translates it to chef-speak (Deserialization), grabs the finished dish (The PyTorch Output), and serves it beautifully on a plate. The investor never has to see the messy PyTorch code, but they get to experience the brilliance of your models.